# NB_ENR_POI_INFO

APPEND `lh_vv_gold.enr.gld_enr_poi_info`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:

def _keys(frame, column):
    return frame.select(F.col(column).alias("poi_id")).where(F.col("poi_id").isNotNull()).distinct()


def _duration_label(sector):
    # RULE-003 — MQ-041, MQ-054. Khớp đúng 11 nhãn. accomodation viết sai rơi ELSE.
    return (F.when(sector == "accommodation", F.lit("Thời gian lưu trú lý tưởng"))
             .when(sector == "f_n_b", F.lit("Thời gian dùng bữa"))
             .when(sector == "attraction", F.lit("Thời gian tham quan"))
             .when(sector == "tour_n_activities", F.lit("Thời lượng hoạt động"))
             .when(sector == "shopping_n_retail", F.lit("Thời gian mua sắm"))
             .when(sector == "transport_route_base", F.lit("Thời gian đưa đón"))
             .when(sector == "transport_on_demand", F.lit("Thời gian thuê phổ biến"))
             .when(sector == "vehicle_rental", F.lit("Thời gian thuê phổ biến"))
             .when(sector == "spa_n_wellness", F.lit("Thời lượng liệu trình"))
             .when(sector == "event", F.lit("Thời lượng chương trình"))
             .otherwise(F.lit("Thời lượng trải nghiệm")))


def _review_json(reviews):
    # COL-048 — MQ-053, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P. RULE-006.
    window = Window.partitionBy("poi_id").orderBy(F.col("time").desc(), F.col("review_id").asc())
    top = (reviews.withColumn("rn", F.row_number().over(window)).where(F.col("rn") <= 20)
           .select("poi_id", F.struct(
               F.col("rn"), F.col("review_id"), F.substring(F.col("text"), 1, 350).alias("review_text"),
               F.col("time"), F.col("published_at")).alias("elem")))
    packed = top.groupBy("poi_id").agg(F.collect_list("elem").alias("elems"))
    payload = F.to_json(F.transform(F.array_sort(F.col("elems")), lambda row: F.struct(
        row.review_id.alias("review_id"), row.review_text.alias("review_text"),
        row.time.alias("time"), row.published_at.alias("published_at"))))
    return packed.select("poi_id", payload.alias("review_json"))


def _blank(value):
    return F.coalesce(value, F.lit(""))


def _destination_context(short_address, district, city, destination_name):
    # COL-043 — MQ-036, MQ-061, docs/context/01_DATA_MODEL.md §4
    # decision.md 2026-10-13: NULL thành chuỗi rỗng, câu vẫn ghép.
    return F.concat(
        F.lit("This POI is located at "),
        _blank(short_address), F.lit(", "),
        _blank(district), F.lit(", "),
        _blank(city), F.lit(", Vietnam (destination area: "),
        _blank(destination_name), F.lit(")."),
    )


def build(ctx: NodeContext) -> DataFrame:
    poi_universe = ctx.input("poi").select("poi_id")
    changed_3p = None
    for role in ("poi", "addr", "contact", "rating", "price", "hours", "review"):
        part = _keys(ctx.changed(role), "poi_id")
        changed_3p = part if changed_3p is None else changed_3p.union(part)
    changed_3p = changed_3p.distinct().join(poi_universe, "poi_id", "left_semi")
    dropped = _keys(ctx.changed("review"), "poi_id").join(poi_universe, "poi_id", "left_anti").count()
    ctx.warn("review_poi_not_in_slv_3p_poi", dropped)
    p = ctx.input("poi").join(changed_3p, "poi_id", "left_semi").alias("p")
    a = ctx.input("addr").alias("a")
    c = ctx.input("contact").alias("c")
    r = ctx.input("rating").alias("r")
    pr = ctx.input("price").alias("pr")
    oh = ctx.input("hours").alias("oh")
    rj = _review_json(ctx.input("review").join(changed_3p, "poi_id", "left_semi")).alias("rj")
    base = (p.join(a, F.col("p.poi_id") == F.col("a.poi_id"), "left")
             .join(c, F.col("p.poi_id") == F.col("c.poi_id"), "left")
             .join(r, F.col("p.poi_id") == F.col("r.poi_id"), "left")
             .join(pr, F.col("p.poi_id") == F.col("pr.poi_id"), "left")
             .join(oh, F.col("p.poi_id") == F.col("oh.poi_id"), "left")
             .join(rj, F.col("p.poi_id") == F.col("rj.poi_id"), "left"))
    dest_name = F.col("a.city")  # [SỬA 19/10] city only
    label = _duration_label(F.col("p.business_sector"))
    third = base.select(
        F.col("p.poi_id").alias("poi_id"),  # COL-001 — MQ-006, MQ-039, MQ-063, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("p.poi_name").alias("poi_name"),  # COL-035 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("p.types_json").alias("types"),  # COL-036 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("p.types_json").alias("subcategory_tags"),  # COL-008 — MQ-049, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("p.business_sector").alias("business_sector"),  # COL-037 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("p.business_category").alias("business_category"),  # COL-038 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("c.website_url").alias("website_url"),  # COL-039 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.address_viet_map").alias("short_address"),  # COL-040 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.district").alias("district"),  # COL-041 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        dest_name.alias("destination_name"),  # COL-042 — MQ-050, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        _destination_context(F.col("a.short_address"), F.col("a.district"), F.col("a.city"), dest_name).alias("destination_context"),
        F.col("r.rating_overall").alias("rating"),  # COL-044 — MQ-035, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("r.rating_count").alias("userRatingCount"),  # COL-045 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("pr.price_level").alias("priceLevel"),  # COL-046 — MQ-033, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("oh.periods_json").alias("opening_hours_text"),  # COL-047 — MQ-037, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.coalesce(F.col("rj.review_json"), F.lit("[]")).alias("review_json"),
        F.to_json(F.create_map(label, F.lit("Không cố định"))).alias("time_to_spend"),  # COL-003 — MQ-048, MQ-061, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.lit(None).cast("string").alias("best_time_type"),  # COL-004 — MQ-009, MQ-061
        F.lit(None).cast("int").alias("best_time_from_month"),  # COL-005 — MQ-010, MQ-061
        F.lit(None).cast("int").alias("best_time_to_month"),  # COL-006 — MQ-010, MQ-061
        F.lit(None).cast("bigint").alias("cost"),  # COL-007 — MQ-061
    )
    # COL-009 — MQ-052, MQ-058. created_at do lib gán batch_ts.
    # COL-010 — MQ-039, MQ-052, MQ-058. updated_at do lib gán batch_ts.
    # decision.md 2026-10-13: gld_enr_poi_info chỉ có nguồn 3rd-party.
    return third


SPEC = NodeSpec(
    trg_tbl="gld_enr_poi_info",
    inputs={
        "poi": "slv_3p_poi", "addr": "slv_3p_poi_address", "contact": "slv_3p_poi_contact",
        "rating": "slv_3p_poi_rating", "price": "slv_3p_poi_price", "hours": "slv_3p_poi_opening_hours",
        "review": "slv_3p_poi_review",
    },
    pk=("poi_id",),
    build=build,
    required_inputs=("poi",),
    write_mode="APPEND",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
